In [1]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

In [2]:
def balance_df(df):  # helper function for csv2df
    normal = df[df.Label == 0]
    attack = df[df.Label == 1]
    # If there are no attack samples, return sorted original
    if len(attack) == 0:
        out = df.sort_values("Time")
        out = out.infer_objects(copy=False)
        return out
    # Downsample to balanced size
    n = min(len(normal), len(attack))
    normal_down = normal.sample(n, random_state=42)
    attack_down = attack.sample(n, random_state=42)
    # Combine and sort
    out = pd.concat([normal_down, attack_down], ignore_index=True)
    out = out.sort_values("Time")
    # Convert dtypes where possible
    out = out.infer_objects(copy=False)
    #  interpolate only numeric columns
    num_cols = out.select_dtypes(include=["number"]).columns
    out[num_cols] = out[num_cols].interpolate().ffill().bfill()
    return out

def slice_by_rows(df, hours=2, hz=20): #helper function for data reduction
    # if 20 Hz sampling → 20 samples per second → 72k samples per hour
    rows_per_hour = hz * 3600
    n = hours * rows_per_hour
    return df.iloc[:n]

In [3]:

def resample_fixed_hz_numpy(df, target_hz=20, gap_sec=1.0):
    """
    Memory-efficient resampling using NumPy for large SynCAN datasets.
    Only numeric signal columns are interpolated.
    """
    times = df["Time"].values.astype(np.float64)
    labels = df["Label"].values.astype(np.float32)

    # Identify sessions by gaps
    gaps = np.diff(times, prepend=times[0])
    session_ids = np.cumsum(gaps > gap_sec)

    # Numeric signals only
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    signal_cols = [c for c in numeric_cols if c not in ["Time", "Label"]]
    signals = df[signal_cols].values.astype(np.float32)

    resampled_list = []

    for s in np.unique(session_ids):
        mask = session_ids == s
        t_chunk = times[mask]
        lbl_chunk = labels[mask]
        sig_chunk = signals[mask]

        if len(t_chunk) < 2:
            continue

        # Uniform timeline
        start, end = t_chunk[0], t_chunk[-1]
        dt = 1 / target_hz
        t_uniform = np.arange(start, end + dt, dt, dtype=np.float32)

        # Interpolate numeric signals
        sig_resampled = np.empty((len(t_uniform), sig_chunk.shape[1]), dtype=np.float32)
        for i in range(sig_chunk.shape[1]):
            sig_resampled[:, i] = np.interp(t_uniform, t_chunk, sig_chunk[:, i])

        # Interpolate labels (nearest)
        idx = np.searchsorted(t_chunk, t_uniform)
        idx[idx >= len(lbl_chunk)] = len(lbl_chunk) - 1
        lbl_resampled = lbl_chunk[idx]
        # Combine
        res_chunk = np.zeros((len(t_uniform), sig_chunk.shape[1] + 2), dtype=np.float32)
        res_chunk[:, :sig_chunk.shape[1]] = sig_resampled
        res_chunk[:, -2] = lbl_resampled
        res_chunk[:, -1] = t_uniform
        resampled_list.append(res_chunk)
    # Concatenate all sessions
    resampled_all = np.vstack(resampled_list)
    # Convert back to DataFrame
    col_names = signal_cols + ["Label", "Time"]
    out = pd.DataFrame(resampled_all, columns=col_names)
    # Add ID as None (optional)
    out["ID"] = None
    return out

In [4]:
def csv2df(dir_path, num_hours=2):
    train_data_frames, test_data_frames = [], []
    testing_files = ["continuous", "flooding", "normal", "plateau", "playback", "suppress"]
    csv_path_train = dir_path + "/train_1.csv" #first one is structured different than the rest
    df_temp = pd.read_csv(csv_path_train, low_memory = False) #read in the csv to a df
    df_temp = slice_by_rows(df_temp, hours=num_hours) 
    df_temp = balance_df(df_temp)      #balance the data
    train_data_frames.append(df_temp) #append it to the dataframe list for concatenation later
    for i in range(2, 5):
        csv_path_train = f"{dir_path}/train_{i}.csv"
        df_temp = pd.read_csv(csv_path_train, header = None, names=["Label",  "Time", "ID", "Signal1",  "Signal2",  "Signal3",  "Signal4"],
                                  dtype={"Label": "int8", "Time": "float32", "ID": "str",
           "Signal1": "float32", "Signal2": "float32",
           "Signal3": "float32", "Signal4": "float32"}, low_memory = False)
        df_temp = slice_by_rows(df_temp, hours=num_hours) 
        df_temp = balance_df(df_temp)
        train_data_frames.append(df_temp)
    for file_suffix in testing_files:
        csv_path_test = f"{dir_path}/test_{file_suffix}.csv"
        df_temp = pd.read_csv(csv_path_test, header=0,  names=["Label",  "Time", "ID", "Signal1",  "Signal2",  "Signal3",  "Signal4"],
            dtype={"Label": "int8", "Time": "float32", "ID": "str",
           "Signal1": "float32", "Signal2": "float32",
           "Signal3": "float32", "Signal4": "float32"}, low_memory = False)
        df_temp = slice_by_rows(df_temp, hours=num_hours) 
        df_temp = balance_df(df_temp) 
        test_data_frames.append(df_temp)
    #concatenate the appropriate test/train dfs in their lists into 1 giant df
    test_df = pd.concat(test_data_frames, ignore_index=True)
    test_df = test_df[test_df["Label"].isin([0,1])]
    train_df = pd.concat(train_data_frames, ignore_index=True)
    return train_df, test_df

In [ ]:
def create_windows(df, window_sec=3, stride_sec=0.5, target_hz=200): #made similar training example windows to ROAD dataset method
    # window size in samples
    win_size = int(window_sec * target_hz)
    stride   = int(stride_sec * target_hz)
    X, y = [], []
    for start in range(0, len(df) - win_size, stride):
        end = start + win_size
        window = df.iloc[start:end]
        # label = label of final timestep in window
        label = window["Label"].iloc[-1]
        features = window.drop(columns=["Label", "ID"])
        X.append(features.to_numpy(dtype=np.float32))
        y.append(label)
    return np.array(X), np.array(y)

In [ ]:
# def main():
# Load all training (nromal) and test (normal + attack) CSVs 
train_data, test_data = csv2df("./SynCAN-DATA", num_hours=1)#num_hours=2)
full = pd.concat([train_data, test_data], ignore_index=True)
full = resample_fixed_hz_numpy(full, target_hz=20)
#irregular timestamps so must handle correctly
X = full.drop(columns=["Label"])
y = full["Label"]   
df = full.reset_index(drop=True)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=y)
X_train_w, y_train_w = create_windows(train_df, window_sec=3, stride_sec=1.5, target_hz=20)
X_val_w, y_val_w = create_windows(val_df, window_sec=3, stride_sec=1.5, target_hz=20) #also, 
X_train_w = np.nan_to_num(X_train_w, nan=0.0, posinf=0.0, neginf=0.0)
X_val_w   = np.nan_to_num(X_val_w,   nan=0.0, posinf=0.0, neginf=0.0)
np.savez(
    "SynCANCleanLabeled30min.npz",
        X_train=X_train_w, y_train=y_train_w,
        X_val=X_val_w, y_val=y_val_w,
        allow_pickle=True
)
# if __name__ == "__main__":
#     main()

In [13]:
#import the datasets to test
def load_data(data_file_name):
    data = np.load(f"Preprocessed_Data/{data_file_name}", allow_pickle=True)
    # Access arrays by their keys
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test  = data["X_val"]
    y_test  = data["y_val"]
    X_train = X_train[..., :-1] #TEMP FIX REMOVE LATER - FIX PREPROCESSING TO GET RID OF STRING COLUMN
    X_test  = X_test[..., :-1] #TEMP FIX REMOVE LATER
    print(X_train.shape)
    print(y_train.shape)
    print(X_test.shape)
    print(y_test.shape)
    data.close()
    return X_train, y_train, X_test, y_test
X_train, y_train, X_test, y_test = load_data("SynCANCleanLabeled2hrs.npz")


(40133, 60, 5)
(40133,)
(10032, 60, 5)
(10032,)


In [23]:
X_test[1], y_test[1]
# y_test[y_test == 1]
# i = 0
for x,y in zip(X_test, y_test):
    i+=1
    if y == 1:
        print(y, "Index: ", i)
        break

1.0 Index:  18


In [45]:
idx_neg = np.where(y_test == 0)[0][0]
idx_pos = np.where(y_test == 1)[0][0]

X_test_small = X_test[[idx_neg, idx_pos]]
y_test_small = y_test[[idx_neg, idx_pos]]

# np.where(np.isnan(X_test_small))

X_test_small.dtype

dtype('O')

In [46]:
X_test_small = X_test_small.astype(np.float32)
X_test_small = np.nan_to_num(X_test_small, nan=0.0, posinf=0.0, neginf=0.0)

In [48]:
print(type(X_test))
print(X_test.dtype)

<class 'numpy.ndarray'>
object


In [47]:
np.isnan(X_test_small).sum()

np.int64(0)

In [40]:
#prepare data for testing in STM32
def save_test_h(X, y, filename="test_data.h"):
    X = X.astype(np.float32)
    y = y.astype(int)

    with open(filename, "w") as f:
        f.write("#pragma once\n\n")

        f.write(f"#define TEST_SAMPLES {X.shape[0]}\n")
        f.write(f"#define TEST_FEATURES {X.shape[1]}\n\n")

        f.write("static const float test_X[TEST_SAMPLES][TEST_FEATURES] = {\n")
        for row in X:
            f.write("  { " + ", ".join(str(v) + "f" for v in row) + " },\n")
        f.write("};\n\n")

        f.write("static const int test_y[TEST_SAMPLES] = {\n")
        f.write("  " + ", ".join(str(v) for v in y) + "\n")
        f.write("};\n")

save_test_h(X_test_small, y_test_small, "test_data_SYNCAN.h")

In [38]:
y_test_small

array([0., 1.], dtype=float32)

In [7]:
# 7. Simple feedforward NN (like INDRA notebook style)
# model = Sequential([
#     Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
#     Dropout(0.3),
#     Dense(64, activation="relu"),
#     Dropout(0.3),
#     Dense(1, activation="sigmoid")
# ])

# model.compile(optimizer=Adam(learning_rate=1e-3),
#               loss="binary_crossentropy",
#               metrics=["accuracy"])


In [8]:

# 8. Train
# history = model.fit(X_train, y_train,
#                     validation_data=(X_val, y_val),
#                     epochs=10,
#                     batch_size=256)


In [9]:

# # 9. Evaluate
# loss, acc = model.evaluate(X_val, y_val)
# print(f"Validation accuracy: {acc:.3f}")